# Ideal PFB Spectral Response

This notebook inspects the ideal PFB-only response already
exposed by `setigen.voltage.PolyphaseFilterbank`. This is the
response from the PFB FIR/window coefficients, not a complete
telescope bandpass and not a sky continuum model.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pfb_response_tools import (
    PFBExperimentConfig,
    detected_intensity_sweep,
    ideal_response,
    local_noise_stats,
    modeled_bandpass_excess_sweep,
    modeled_bandpass_summary,
    noise_overlay_summary,
    normalize_column,
    run_spectrogram,
    tone_response_sweep,
)

plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})


In [ ]:
config = PFBExperimentConfig()
response = ideal_response(config)

summary = {
    "fine channels": len(response),
    "min / mean": response.min(),
    "max / mean": response.max(),
    "max / min": response.max() / response.min(),
    "df Hz": config.df,
    "coarse channel bandwidth Hz": config.chan_bw,
}
summary


In [ ]:
fine = np.arange(config.fftlength)
fine_offset = (fine - config.fftlength / 2) * config.df / 1e3

fig, ax = plt.subplots()
ax.plot(fine_offset, response, lw=2, color="tab:blue")
ax.axvline(0, color="0.25", lw=1, alpha=0.7)
ax.set_xlabel("Fine-channel offset from coarse-channel center (kHz)")
ax.set_ylabel("PFB power response / mean")
ax.set_title("Ideal PFB response within one coarse channel")
display(fig)
plt.close(fig)


In [ ]:
tiled = ideal_response(config, num_chans=4)
flat_index = np.arange(len(tiled))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(flat_index, tiled, lw=1.5, color="tab:purple")
for boundary in range(config.fftlength, len(tiled), config.fftlength):
    ax.axvline(boundary, color="0.3", lw=0.8, alpha=0.5)
ax.set_xlabel("Flattened fine-channel index")
ax.set_ylabel("PFB power response / mean")
ax.set_title("Response tiled across coarse channels")
display(fig)
plt.close(fig)


The important feature is the repeated coarse-channel structure:
the ideal response is high through much of the coarse-channel
interior and lower near the coarse-channel edges. Any SNR
estimate over wide spectral regions should not assume a single
stationary background variance.
